# Field-Ready Plant Disease Detection

Models trained on **PlantVillage** (single leaves photographed on plain lab backgrounds) routinely score 97-99%.
Farmers, though, photograph leaves in the field: other leaves, soil, shadows, blur.
This notebook **measures how much accuracy is lost on real field photos (PlantDoc)** and **reduces that gap**.

| Model | What changes |
|---|---|
| **M1 baseline** | EfficientNetV2-B0, transfer learning on PlantVillage, light augmentation |
| **M2 robust** | same, with strong "field-like" augmentation (crops, rotation, colour, occlusion, noise) |
| **M3 field** | M2 fine-tuned on PlantVillage + PlantDoc *train* images |

Every model is evaluated on the **PlantVillage test split** and the **PlantDoc test split** (never used for training or model selection).
Then: Grad-CAM explainability, TFLite export for phones, and files for a Gradio demo.

## How to run on Kaggle
1. **Add Input** -> search **"plantvillage dataset"** -> add `abdallahalidev/plantvillage-dataset`.
2. **Settings -> Accelerator: GPU T4 x1** (or P100). **Internet: ON** (PlantDoc is downloaded from GitHub).
3. **Save Version -> Save & Run All (Commit)**. Takes about 2-3 hours. Each model is saved as it finishes, so a re-run skips finished stages.
4. When it finishes, download **`outputs.zip`** from the Output tab.

In [ ]:
import os, glob, json, time, math, re, shutil, subprocess

import numpy as np
import tensorflow as tf
import keras
from keras import layers
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, confusion_matrix

SEED = 42
keras.utils.set_random_seed(SEED)

FAST = os.environ.get("FAST") == "1"          # tiny smoke-test mode (not used on Kaggle)
WORK = os.environ.get("WORK_DIR", "/kaggle/working")
IMG = 64 if FAST else 224
BATCH = 8 if FAST else 32
WEIGHTS = None if FAST else "imagenet"
EPOCHS = dict(frozen=1, finetune=1, field=1) if FAST else dict(frozen=2, finetune=4, field=6)
FIELD_STEPS = 2 if FAST else 300               # steps per epoch for the mixed field fine-tune
MAX_STEPS = 3 if FAST else None                # cap steps per epoch (smoke test only)

MODELS, RESULTS, APP = f"{WORK}/models", f"{WORK}/results", f"{WORK}/app"
for d in (MODELS, RESULTS, APP):
    os.makedirs(d, exist_ok=True)

GPUS = tf.config.list_physical_devices("GPU")
if GPUS and not FAST:
    keras.mixed_precision.set_global_policy("mixed_float16")   # ~2x faster on T4/P100
print(f"TF {tf.__version__} | Keras {keras.__version__} | GPUs: {len(GPUS)} | image {IMG} | batch {BATCH}")

## 1. PlantVillage: lab images (train / validation / test = 80 / 10 / 10)

In [ ]:
def find_plantvillage():
    if os.environ.get("PV_DIR"):
        return os.environ["PV_DIR"]
    hits = glob.glob("/kaggle/input/**/Tomato___healthy", recursive=True)
    parents = sorted({os.path.dirname(h) for h in hits})
    # prefer the original colour images; avoid grayscale / segmented / pre-augmented copies
    parents.sort(key=lambda p: ("color" not in p.lower(),
                                "segment" in p.lower() or "gray" in p.lower(),
                                "augmented" in p.lower(), len(p)))
    if not parents:
        raise FileNotFoundError("PlantVillage not found. Add Input -> 'abdallahalidev/plantvillage-dataset'.")
    return parents[0]

PV_DIR = find_plantvillage()
CLASSES = sorted(d for d in os.listdir(PV_DIR) if os.path.isdir(os.path.join(PV_DIR, d)))
NUM_CLASSES = len(CLASSES)
EXTS = (".jpg", ".jpeg", ".png")

paths, labels = [], []
for i, c in enumerate(CLASSES):
    for f in sorted(os.listdir(os.path.join(PV_DIR, c))):
        if f.lower().endswith(EXTS):
            paths.append(os.path.join(PV_DIR, c, f))
            labels.append(i)
paths, labels = np.array(paths), np.array(labels)

p_train, p_tmp, y_train, y_tmp = train_test_split(paths, labels, test_size=0.2, stratify=labels, random_state=SEED)
p_val, p_test, y_val, y_test = train_test_split(p_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=SEED)
print(f"PlantVillage: {PV_DIR}")
print(f"{NUM_CLASSES} classes | train {len(p_train)} | val {len(p_val)} | test {len(p_test)}")

## 2. PlantDoc: real field photos
PlantDoc (Singh et al., 2020) has leaves photographed in real conditions. Its class names differ from PlantVillage,
so each PlantDoc folder is mapped to the matching PlantVillage class. Folders with no PlantVillage equivalent are skipped and listed.

In [ ]:
def norm(s):
    return re.sub(r"[^a-z]", "", s.lower())

# PlantDoc folder name (normalised) -> PlantVillage class
PD_MAP = {
    "applescableaf": "Apple___Apple_scab",
    "appleleaf": "Apple___healthy",
    "applerustleaf": "Apple___Cedar_apple_rust",
    "bellpepperleaf": "Pepper,_bell___healthy",
    "bellpepperleafspot": "Pepper,_bell___Bacterial_spot",
    "blueberryleaf": "Blueberry___healthy",
    "cherryleaf": "Cherry_(including_sour)___healthy",
    "corngrayleafspot": "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
    "cornleafblight": "Corn_(maize)___Northern_Leaf_Blight",
    "cornrustleaf": "Corn_(maize)___Common_rust_",
    "peachleaf": "Peach___healthy",
    "potatoleafearlyblight": "Potato___Early_blight",
    "potatoleaflateblight": "Potato___Late_blight",
    "raspberryleaf": "Raspberry___healthy",
    "soyabeanleaf": "Soybean___healthy",
    "squashpowderymildewleaf": "Squash___Powdery_mildew",
    "strawberryleaf": "Strawberry___healthy",
    "tomatoearlyblightleaf": "Tomato___Early_blight",
    "tomatoseptorialeafspot": "Tomato___Septoria_leaf_spot",
    "tomatoleaf": "Tomato___healthy",
    "tomatoleafbacterialspot": "Tomato___Bacterial_spot",
    "tomatoleaflateblight": "Tomato___Late_blight",
    "tomatoleafmosaicvirus": "Tomato___Tomato_mosaic_virus",
    "tomatoleafyellowvirus": "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "tomatomoldleaf": "Tomato___Leaf_Mold",
    "tomatotwospottedspidermitesleaf": "Tomato___Spider_mites Two-spotted_spider_mite",
    "grapeleaf": "Grape___healthy",
    "grapeleafblackrot": "Grape___Black_rot",
}

def get_plantdoc():
    if os.environ.get("PD_DIR"):
        return os.environ["PD_DIR"]
    hits = glob.glob("/kaggle/input/**/train/Tomato leaf", recursive=True)
    if hits:
        return os.path.dirname(os.path.dirname(hits[0]))
    dest = f"{WORK}/PlantDoc-Dataset"
    if not os.path.isdir(dest):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/pratikkayal/PlantDoc-Dataset", dest], check=True)
    return dest

def load_plantdoc(split):
    """Loads one PlantDoc split into memory, resized, with PlantVillage label ids."""
    root = os.path.join(PD_DIR, split)
    X, y, skipped, bad = [], [], set(), 0
    for folder in sorted(os.listdir(root)):
        full = os.path.join(root, folder)
        if not os.path.isdir(full):
            continue
        target = PD_MAP.get(norm(folder))
        if target not in CLASSES:
            skipped.add(folder)
            continue
        for f in sorted(os.listdir(full)):
            if not f.lower().endswith(EXTS):
                continue
            try:
                im = Image.open(os.path.join(full, f)).convert("RGB").resize((IMG, IMG), Image.BILINEAR)
                X.append(np.asarray(im, dtype=np.uint8))
                y.append(CLASSES.index(target))
            except Exception:
                bad += 1
    print(f"PlantDoc {split}: {len(X)} images | {len(set(y))} classes | unreadable {bad}")
    if skipped:
        print("  skipped (no PlantVillage match):", sorted(skipped))
    return np.stack(X), np.array(y)

PD_DIR = get_plantdoc()
Xpd_train_all, ypd_train_all = load_plantdoc("train")
Xpd_test, ypd_test = load_plantdoc("test")

# hold out 15% of PlantDoc *train* as validation for the field fine-tune (the test split is never touched)
try:
    Xpd_tr, Xpd_va, ypd_tr, ypd_va = train_test_split(Xpd_train_all, ypd_train_all, test_size=0.15,
                                                      stratify=ypd_train_all, random_state=SEED)
except ValueError:  # a class too small to stratify
    Xpd_tr, Xpd_va, ypd_tr, ypd_va = train_test_split(Xpd_train_all, ypd_train_all, test_size=0.15,
                                                      random_state=SEED)
print(f"PlantDoc field fine-tune: train {len(Xpd_tr)} | val {len(Xpd_va)} | test {len(Xpd_test)}")

## 3. Input pipelines and augmentation
* **light**: horizontal flip only (what most PlantVillage notebooks do)
* **strong**: random crop/zoom, rotation, flips, shifts, colour jitter, a random occluding patch and sensor noise,
  to imitate what changes between a lab photo and a phone photo in a field

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def load_path(path, label):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG, IMG))
    return img, label

def pv_items(p, y, shuffle=False, repeat=False):
    ds = tf.data.Dataset.from_tensor_slices((p, y))
    if shuffle:
        ds = ds.shuffle(len(p), seed=SEED, reshuffle_each_iteration=True)
    if repeat:
        ds = ds.repeat()
    return ds.map(load_path, num_parallel_calls=AUTOTUNE)

def np_items(X, y, shuffle=False, repeat=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle:
        ds = ds.shuffle(len(X), seed=SEED, reshuffle_each_iteration=True)
    if repeat:
        ds = ds.repeat()
    return ds.map(lambda a, b: (tf.cast(a, tf.float32), b), num_parallel_calls=AUTOTUNE)

def field_like(img, label):
    """Per-image colour jitter, a random occluding patch and noise (input range 0-255)."""
    x = img / 255.0
    x = tf.image.random_brightness(x, 0.25)
    x = tf.image.random_contrast(x, 0.6, 1.4)
    x = tf.image.random_saturation(x, 0.6, 1.4)
    x = tf.image.random_hue(x, 0.05)
    x = tf.clip_by_value(x, 0.0, 1.0)
    # random square occlusion (another leaf, a finger, a shadow) with probability 0.5
    size = IMG // 4
    cy = tf.random.uniform([], 0, IMG, dtype=tf.int32)
    cx = tf.random.uniform([], 0, IMG, dtype=tf.int32)
    yy, xx = tf.meshgrid(tf.range(IMG), tf.range(IMG), indexing="ij")
    box = tf.logical_and(tf.abs(yy - cy) < size // 2, tf.abs(xx - cx) < size // 2)
    use = tf.random.uniform([]) < 0.5
    fill = tf.random.uniform([], 0.2, 0.8)
    x = tf.where(tf.logical_and(box, use)[..., None], fill, x)
    # sensor noise with probability 0.3
    noise = tf.random.normal(tf.shape(x), stddev=0.03)
    x = tf.where(tf.random.uniform([]) < 0.3, x + noise, x)
    return tf.clip_by_value(x, 0.0, 1.0) * 255.0, label

LIGHT_GEO = keras.Sequential([layers.RandomFlip("horizontal")], name="light_aug")
STRONG_GEO = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15),
    layers.RandomZoom((-0.3, 0.1)),
    layers.RandomTranslation(0.1, 0.1),
], name="strong_aug")

def finalize(items, aug=None):
    if aug == "strong":
        items = items.map(field_like, num_parallel_calls=AUTOTUNE)
    ds = items.batch(BATCH)
    if aug in ("light", "strong"):
        geo = LIGHT_GEO if aug == "light" else STRONG_GEO
        ds = ds.map(lambda x, y: (geo(x, training=True), y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

# quick look at what "strong" augmentation produces
sample = next(iter(finalize(pv_items(p_train[:8], y_train[:8]), "strong")))
plt.figure(figsize=(12, 3))
for i in range(min(6, len(sample[0]))):
    plt.subplot(1, 6, i + 1)
    plt.imshow(sample[0][i].numpy().astype("uint8"))
    plt.axis("off")
plt.suptitle("Strong (field-like) augmentation")
plt.savefig(f"{RESULTS}/augmentation_examples.png", bbox_inches="tight")
plt.show()

## 4. Model: EfficientNetV2-B0 with transfer learning
The original version of this project used VGG16 (~14.7M parameters in the convolutional base alone).
EfficientNetV2-B0 is ~5.9M parameters, faster, more accurate on ImageNet, and small enough for phones.
Training happens in two stages: (1) train only the new head with the base frozen, (2) unfreeze the top 40% of the base
(BatchNorm layers stay frozen) and fine-tune with a lower learning rate.

In [ ]:
HEAD = {"gap", "head_dropout", "pred"}

def build_model(weights=WEIGHTS):
    base = keras.applications.EfficientNetV2B0(include_top=False, weights=weights,
                                               input_shape=(IMG, IMG, 3), include_preprocessing=True)
    x = layers.GlobalAveragePooling2D(name="gap")(base.output)
    x = layers.Dropout(0.3, name="head_dropout")(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32", name="pred")(x)
    return keras.Model(base.input, out, name="effnetv2b0_plant")

def set_trainable(model, mode):
    base_layers = [l for l in model.layers if l.name not in HEAD and not isinstance(l, layers.InputLayer)]
    cut = int(len(base_layers) * 0.6)
    for i, l in enumerate(base_layers):
        if mode == "frozen":
            l.trainable = False
        else:  # fine-tune the top 40%, keep BatchNorm frozen
            l.trainable = i >= cut and not isinstance(l, layers.BatchNormalization)
    for name in HEAD:
        model.get_layer(name).trainable = True

def compile_model(model, lr):
    model.compile(optimizer=keras.optimizers.Adam(lr),
                  loss=keras.losses.SparseCategoricalCrossentropy(),
                  metrics=["accuracy", keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3")])

m = build_model(weights=None)
vgg = keras.applications.VGG16(include_top=False, weights=None, input_shape=(IMG, IMG, 3))
print(f"EfficientNetV2-B0 classifier: {m.count_params() / 1e6:.1f}M params | VGG16 conv base alone: {vgg.count_params() / 1e6:.1f}M")
del m, vgg

## 5. Training helpers (resumable) and evaluation

In [ ]:
HISTORY = {}

def save_history(name, *hists):
    merged = {}
    for h in hists:
        for k, v in h.history.items():
            merged.setdefault(k, []).extend(float(x) for x in v)
    HISTORY[name] = merged
    with open(f"{RESULTS}/history_{name}.json", "w") as f:
        json.dump(merged, f, indent=2)

def load_history(name):
    f = f"{RESULTS}/history_{name}.json"
    if os.path.exists(f):
        HISTORY[name] = json.load(open(f))

def train_on_plantvillage(name, aug):
    path = f"{MODELS}/{name}.keras"
    if os.path.exists(path):
        print(f"[{name}] found {path} - skipping training")
        load_history(name)
        return keras.models.load_model(path)
    t0 = time.time()
    model = build_model()
    train_ds = finalize(pv_items(p_train, y_train, shuffle=True), aug)
    val_ds = finalize(pv_items(p_val, y_val))
    print(f"[{name}] stage 1: head only")
    set_trainable(model, "frozen")
    compile_model(model, 1e-3)
    h1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS["frozen"],
                   steps_per_epoch=MAX_STEPS, validation_steps=MAX_STEPS, verbose=2)
    print(f"[{name}] stage 2: fine-tune top of the base")
    set_trainable(model, "finetune")
    compile_model(model, 1e-4)
    ckpt = keras.callbacks.ModelCheckpoint(path, monitor="val_accuracy", save_best_only=True)
    h2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS["finetune"],
                   steps_per_epoch=MAX_STEPS, validation_steps=MAX_STEPS, callbacks=[ckpt], verbose=2)
    save_history(name, h1, h2)
    print(f"[{name}] done in {(time.time() - t0) / 60:.1f} min")
    return keras.models.load_model(path)

def wilson(k, n, z=1.96):
    """95% confidence interval for an accuracy of k/n."""
    if n == 0:
        return (0.0, 0.0)
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (round(c - h, 4), round(c + h, 4))

def score(y_true, probs):
    y_pred = probs.argmax(1)
    top3 = np.argsort(-probs, axis=1)[:, :3]
    k = int((y_pred == y_true).sum())
    return {
        "n": int(len(y_true)),
        "accuracy": round(k / len(y_true), 4),
        "accuracy_95ci": wilson(k, len(y_true)),
        "top3_accuracy": round(float(np.mean([t in row for t, row in zip(y_true, top3)])), 4),
        "macro_f1": round(float(f1_score(y_true, y_pred, labels=np.unique(y_true), average="macro")), 4),
    }

METRICS_PATH = f"{RESULTS}/metrics.json"
METRICS = json.load(open(METRICS_PATH)) if os.path.exists(METRICS_PATH) else {}
PROBS = {}

def evaluate(model, name):
    pv_ds = finalize(pv_items(p_test, y_test))
    if MAX_STEPS:
        pv_ds = pv_ds.take(MAX_STEPS)
    pv_probs = model.predict(pv_ds, verbose=0)
    y_pv = y_test[: len(pv_probs)]
    pd_probs = model.predict(finalize(np_items(Xpd_test, ypd_test)), verbose=0)
    PROBS[name] = {"pv": (y_pv, pv_probs), "pd": (ypd_test, pd_probs)}
    METRICS[name] = {"plantvillage_test": score(y_pv, pv_probs), "plantdoc_test": score(ypd_test, pd_probs)}
    with open(METRICS_PATH, "w") as f:
        json.dump(METRICS, f, indent=2)
    a, b = METRICS[name]["plantvillage_test"], METRICS[name]["plantdoc_test"]
    print(f"[{name}] PlantVillage test acc {a['accuracy']:.3f} (n={a['n']}) | "
          f"PlantDoc field acc {b['accuracy']:.3f} (95% CI {b['accuracy_95ci']}, n={b['n']}) | "
          f"field top-3 {b['top3_accuracy']:.3f}")

## 6. M1 baseline: PlantVillage, light augmentation

In [ ]:
m1 = train_on_plantvillage("m1_baseline", aug="light")
evaluate(m1, "m1_baseline")

## 7. M2 robust: PlantVillage, strong field-like augmentation

In [ ]:
m2 = train_on_plantvillage("m2_robust_aug", aug="strong")
evaluate(m2, "m2_robust_aug")

## 8. M3 field: fine-tune M2 on a 50/50 mix of PlantVillage and PlantDoc train images
Model selection uses the PlantDoc **validation** split carved from PlantDoc train. The PlantDoc test split stays unseen.

In [ ]:
def train_field(name="m3_field_finetune", base_name="m2_robust_aug"):
    path = f"{MODELS}/{name}.keras"
    if os.path.exists(path):
        print(f"[{name}] found {path} - skipping training")
        load_history(name)
        return keras.models.load_model(path)
    t0 = time.time()
    model = keras.models.load_model(f"{MODELS}/{base_name}.keras")
    set_trainable(model, "finetune")
    compile_model(model, 3e-5)
    mixed = tf.data.Dataset.sample_from_datasets(
        [pv_items(p_train, y_train, shuffle=True, repeat=True),
         np_items(Xpd_tr, ypd_tr, shuffle=True, repeat=True)],
        weights=[0.5, 0.5], seed=SEED)
    train_ds = finalize(mixed, "strong")
    val_ds = finalize(np_items(Xpd_va, ypd_va))
    ckpt = keras.callbacks.ModelCheckpoint(path, monitor="val_accuracy", save_best_only=True)
    h = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS["field"],
                  steps_per_epoch=FIELD_STEPS, callbacks=[ckpt], verbose=2)
    save_history(name, h)
    print(f"[{name}] done in {(time.time() - t0) / 60:.1f} min")
    return keras.models.load_model(path)

m3 = train_field()
evaluate(m3, "m3_field_finetune")

## 9. Results: the lab-to-field gap

In [ ]:
names = ["m1_baseline", "m2_robust_aug", "m3_field_finetune"]
pretty = {"m1_baseline": "M1 baseline", "m2_robust_aug": "M2 robust aug", "m3_field_finetune": "M3 field fine-tune"}

lines = ["| Model | PlantVillage test acc | PlantDoc field acc (95% CI) | Field top-3 | Field macro-F1 |",
         "|---|---|---|---|---|"]
for n in names:
    a, b = METRICS[n]["plantvillage_test"], METRICS[n]["plantdoc_test"]
    lo, hi = b["accuracy_95ci"]
    lines.append(f"| {pretty[n]} | {a['accuracy']:.1%} | {b['accuracy']:.1%} ({lo:.1%}-{hi:.1%}) | "
                 f"{b['top3_accuracy']:.1%} | {b['macro_f1']:.3f} |")
table = "\n".join(lines)
print(table)
with open(f"{RESULTS}/results_table.md", "w") as f:
    f.write(table + f"\n\nPlantVillage test n={METRICS['m1_baseline']['plantvillage_test']['n']}, "
                    f"PlantDoc test n={METRICS['m1_baseline']['plantdoc_test']['n']}.\n")

x = np.arange(len(names))
pv_acc = [METRICS[n]["plantvillage_test"]["accuracy"] for n in names]
pd_acc = [METRICS[n]["plantdoc_test"]["accuracy"] for n in names]
plt.figure(figsize=(8, 4.5))
plt.bar(x - 0.2, pv_acc, 0.4, label="PlantVillage test (lab)")
plt.bar(x + 0.2, pd_acc, 0.4, label="PlantDoc test (field)")
for i in range(len(names)):
    plt.text(x[i] - 0.2, pv_acc[i] + 0.01, f"{pv_acc[i]:.0%}", ha="center")
    plt.text(x[i] + 0.2, pd_acc[i] + 0.01, f"{pd_acc[i]:.0%}", ha="center")
plt.xticks(x, [pretty[n] for n in names])
plt.ylim(0, 1.1)
plt.ylabel("Accuracy")
plt.title("Lab accuracy vs field accuracy")
plt.legend()
plt.savefig(f"{RESULTS}/accuracy_gap.png", bbox_inches="tight", dpi=150)
plt.show()

# per-class reports for the final model
final = "m3_field_finetune"
y_pd, p_pd = PROBS[final]["pd"]
present = np.unique(y_pd)
report_pd = classification_report(y_pd, p_pd.argmax(1), labels=present,
                                  target_names=[CLASSES[i] for i in present], zero_division=0)
y_pv, p_pv = PROBS[final]["pv"]
report_pv = classification_report(y_pv, p_pv.argmax(1), labels=np.unique(y_pv),
                                  target_names=[CLASSES[i] for i in np.unique(y_pv)], zero_division=0)
open(f"{RESULTS}/report_plantdoc_final.txt", "w").write(report_pd)
open(f"{RESULTS}/report_plantvillage_final.txt", "w").write(report_pv)
print(report_pd)

cm = confusion_matrix(y_pd, p_pd.argmax(1), labels=present)
short = [CLASSES[i].replace("___", ": ").replace("_", " ")[:28] for i in present]
plt.figure(figsize=(12, 10))
plt.imshow(cm, cmap="Blues")
plt.xticks(range(len(present)), short, rotation=90, fontsize=7)
plt.yticks(range(len(present)), short, fontsize=7)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("PlantDoc field test - confusion matrix (final model)")
plt.colorbar()
plt.savefig(f"{RESULTS}/plantdoc_confusion_final.png", bbox_inches="tight", dpi=150)
plt.show()

for n in names:
    if n in HISTORY and "val_accuracy" in HISTORY[n]:
        plt.plot(HISTORY[n]["val_accuracy"], label=pretty[n])
if HISTORY:
    plt.xlabel("epoch")
    plt.ylabel("validation accuracy")
    plt.legend()
    plt.title("Validation accuracy during training")
    plt.savefig(f"{RESULTS}/training_curves.png", bbox_inches="tight", dpi=150)
    plt.show()

## 10. Grad-CAM: where is the model looking?
Grad-CAM highlights the image regions that drove the prediction. A good model should look at the diseased spots on the leaf,
not at the background.

In [ ]:
def last_conv_name(model):
    try:
        return model.get_layer("top_activation").name
    except ValueError:
        for l in reversed(model.layers):
            if len(l.output.shape) == 4:
                return l.name

def gradcam(model, img):
    """img: float32 (IMG, IMG, 3) in 0-255. Returns (heatmap 0-1, predicted class, confidence)."""
    grad_model = keras.Model(model.input, [model.get_layer(last_conv_name(model)).output, model.output])
    x = tf.convert_to_tensor(img[None], tf.float32)
    with tf.GradientTape() as tape:
        conv, preds = grad_model(x, training=False)
        idx = int(tf.argmax(preds[0]))
        target = preds[:, idx]
    grads = tf.cast(tape.gradient(target, conv), tf.float32)
    conv = tf.cast(conv, tf.float32)
    weights = tf.reduce_mean(grads, axis=(1, 2))
    cam = tf.nn.relu(tf.reduce_sum(conv[0] * weights[0], axis=-1))
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    cam = tf.image.resize(cam[..., None], (IMG, IMG))[..., 0].numpy()
    return cam, idx, float(preds[0, idx])

def label_short(i):
    return CLASSES[i].replace("___", ": ").replace("_", " ")[:30]

rng = np.random.default_rng(SEED)
pick = rng.choice(len(Xpd_test), size=min(6, len(Xpd_test)), replace=False)
fig, axes = plt.subplots(len(pick), 3, figsize=(11, 3.4 * len(pick)))
axes = np.atleast_2d(axes)
for r, i in enumerate(pick):
    img = Xpd_test[i].astype("float32")
    axes[r, 0].imshow(Xpd_test[i])
    axes[r, 0].set_title(f"True: {label_short(ypd_test[i])}", fontsize=8)
    for c, (mdl, nm) in enumerate([(m1, "M1 baseline"), (m3, "M3 field")], start=1):
        cam, p, conf = gradcam(mdl, img)
        ok = "correct" if p == ypd_test[i] else "wrong"
        axes[r, c].imshow(Xpd_test[i])
        axes[r, c].imshow(cam, cmap="jet", alpha=0.45)
        axes[r, c].set_title(f"{nm}: {label_short(p)} ({conf:.0%}, {ok})", fontsize=8)
    for a in axes[r]:
        a.axis("off")
plt.tight_layout()
plt.savefig(f"{RESULTS}/gradcam_baseline_vs_field.png", bbox_inches="tight", dpi=130)
plt.show()

## 11. Export for phones (TFLite) and for the demo app

In [ ]:
# Rebuild the final model in float32 (mixed precision is only for GPU training) and copy the weights.
keras.mixed_precision.set_global_policy("float32")
export_model = build_model(weights=None)
export_model.set_weights(m3.get_weights())
export_model.save(f"{APP}/model.keras")

saved_dir = f"{MODELS}/saved_model"
try:
    try:
        export_model.export(saved_dir, verbose=False)    # Keras 3
    except TypeError:
        export_model.export(saved_dir)
    converter = tf.lite.TFLiteConverter.from_saved_model(saved_dir)
except AttributeError:
    converter = tf.lite.TFLiteConverter.from_keras_model(export_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]       # dynamic-range quantization (int8 weights)
tflite_bytes = converter.convert()
open(f"{APP}/model.tflite", "wb").write(tflite_bytes)

try:
    from ai_edge_litert.interpreter import Interpreter   # newer runtime name for TFLite
except ImportError:
    Interpreter = tf.lite.Interpreter
interp = Interpreter(model_content=tflite_bytes)
interp.allocate_tensors()
inp, out = interp.get_input_details()[0], interp.get_output_details()[0]

def tflite_predict(X):
    probs, times = [], []
    for img in X:
        interp.set_tensor(inp["index"], img[None].astype(np.float32))
        t = time.perf_counter()
        interp.invoke()
        times.append(time.perf_counter() - t)
        probs.append(interp.get_tensor(out["index"])[0])
    return np.array(probs), float(np.mean(times) * 1000)

tfl_probs, tfl_ms = tflite_predict(Xpd_test)
METRICS["tflite_int8_weights"] = {
    "size_mb": round(len(tflite_bytes) / 1e6, 2),
    "keras_size_mb": round(os.path.getsize(f"{APP}/model.keras") / 1e6, 2),
    "plantdoc_test": score(ypd_test, tfl_probs),
    "mean_latency_ms_kaggle_cpu": round(tfl_ms, 1),
}
json.dump(METRICS, open(METRICS_PATH, "w"), indent=2)
print(json.dumps(METRICS["tflite_int8_weights"], indent=2))

labels_out = []
for i, c in enumerate(CLASSES):
    plant, _, disease = c.partition("___")
    labels_out.append({"id": i, "class": c,
                       "plant": plant.replace("_", " ").replace(",", "").replace("(including sour)", "").replace("(maize)", "").strip(),
                       "disease": disease.replace("_", " ").strip() or "unknown",
                       "healthy": disease.lower() == "healthy"})
json.dump({"image_size": IMG, "classes": labels_out}, open(f"{APP}/labels.json", "w"), indent=2)

## 12. Package outputs

In [ ]:
pkg = f"{WORK}/outputs"
shutil.rmtree(pkg, ignore_errors=True)
shutil.copytree(RESULTS, f"{pkg}/results")
shutil.copytree(APP, f"{pkg}/app")
shutil.make_archive(f"{WORK}/outputs", "zip", pkg)
print("Download outputs.zip from the Output tab. Contents:")
for root, _, files in os.walk(pkg):
    for f in files:
        print(" ", os.path.relpath(os.path.join(root, f), pkg))
print("\n" + table)